# MYOREHAB Project: Signal Processing Template

Welcome to the MYOREHAB signal processing environment. This notebook serves as a standardized template for researchers to apply biomechanical and signal processing pipelines to the dataset.

**How to use this template:**
1. **Infrastructure (Do not modify):** Sections 0, 1, 2, and 4 are maintained by the data engineering team. They handle data loading, database connections, and standardizing metadata (e.g., assigning column names, generating time vectors).
2. **Researcher Sandbox (Your workspace):** Section 3 is where you should implement your specific processing algorithms (Notch filters, bandpass filters, envelope extraction, etc.).

## Section 0: Prerequisites
Before running this notebook, ensure you have the required libraries installed in your environment. 
If you are using `uv`, you can add them to your project. Alternatively, you can install them via pip by uncommenting the cell below.

In [ ]:
# Uncomment and run this line if you need to install the dependencies in a standard pip environment:
# !pip install pandas numpy scipy matplotlib

## Section 1: Imports

In [ ]:
# ==============================================================================
# SECTION 1: IMPORTS
# ==============================================================================
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from pathlib import Path

print("Libraries imported successfully.")

## Section 2: Databases

In [ ]:
# ==============================================================================
# SECTION 2: DATABASE CONNECTION & METADATA EXTRACTION
# ==============================================================================
# Define the path to the relational database
DB_PATH = Path("../data/project.db")

def get_subject_metadata(subject_id):
    """
    Queries the database and retrieves metadata and file paths for a specific subject.
    """
    if not DB_PATH.exists():
        print(f"Warning: Database not found at {DB_PATH}. Using dummy metadata for testing.")
        # Fallback for testing purposes if DB is not connected yet
        return pd.DataFrame([{
            'ruta_archivo': '../data/raw/emg_S001_111.csv',
            'frecuencia_muestreo_hz': 2052.0
        }])
        
    conn = sqlite3.connect(DB_PATH)
    query = """
    SELECT 
        r.recording_id, s.subject_id, s.diagnostico,
        t.nombre_tarea, e.modelo AS equipo,
        r.frecuencia_muestreo_hz, r.ruta_archivo
    FROM recording r
    JOIN subjects s ON r.subject_id = s.subject_id
    JOIN tasks t ON r.task_id = t.task_id
    JOIN equipment e ON r.equipment_id = e.equipment_id
    WHERE s.subject_id = ?
    """
    df = pd.read_sql_query(query, conn, params=(subject_id,))
    conn.close()
    return df

# Example: Fetch records for Subject 1
df_metadata = get_subject_metadata(subject_id=1)
display(df_metadata.head())

## Section 3: Signal Processing Functions (Researcher Area)
**Instructions for researchers:** Please fill in the logic for the following functions. The data engineering pipeline will automatically pass the raw signals and the corresponding sampling frequency (`fs`) to these functions.

In [ ]:
# ==============================================================================
# SECTION 3: SIGNAL PROCESSING FUNCTIONS
# ==============================================================================

def apply_notch_filter(signal_data, fs=2052.0, freq_to_remove=60.0, Q=30.0):
    """
    Applies a Notch filter to remove power-line interference.
    
    Parameters:
    -----------
    signal_data : array-like (Numpy array or Pandas Series with raw EMG)
    fs : float (Sampling frequency in Hz)
    freq_to_remove : float (Frequency to remove, e.g., 60.0 Hz in Brazil)
    Q : float (Quality factor)
        
    Returns:
    --------
    filtered_signal : array-like
    
    TODO (For Researchers):
    - Implement scipy.signal.iirnotch
    - Apply scipy.signal.filtfilt
    """
    # === RESEARCHER CODE HERE ===
    # Example implementation:
    b, a = signal.iirnotch(w0=freq_to_remove, Q=Q, fs=fs)
    filtered_signal = signal.filtfilt(b, a, signal_data)
    
    return filtered_signal


def apply_bandpass_filter(signal_data, fs=2052.0, lowcut=20.0, highcut=450.0, order=4):
    """
    Applies a Butterworth bandpass filter to isolate the EMG frequency band.
    
    TODO (For Researchers):
    - Implement scipy.signal.butter
    - Apply scipy.signal.filtfilt
    """
    # === RESEARCHER CODE HERE ===
    filtered_signal = signal_data # Placeholder
    return filtered_signal


def extract_emg_envelope(signal_data, fs=2052.0, method='rms', window_size=50):
    """
    Extracts the envelope of the EMG signal using RMS or low-pass filtering.
    
    TODO (For Researchers):
    - Rectify the signal (absolute value)
    - Apply moving average / RMS window or low-pass filter (e.g., 5-10 Hz)
    """
    # === RESEARCHER CODE HERE ===
    envelope_signal = signal_data # Placeholder
    return envelope_signal

## Section 4: Automated Loading and Pipeline Application
This section takes the metadata, loads the heavy CSV/EDF files, standardizes the column headers, generates the time vector, applies the researcher's functions, and plots a comparative visualization.

In [ ]:
# ==============================================================================
# SECTION 4: AUTOMATED PIPELINE & VISUALIZATION
# ==============================================================================

def process_and_visualize(metadata_row):
    """
    Loads raw data, standardizes it, applies researcher functions, and visualizes.
    """
    filepath = metadata_row['ruta_archivo']
    fs = metadata_row['frecuencia_muestreo_hz']
    
    print(f"Loading data from: {filepath} (Fs = {fs} Hz)")
    
    try:
        # 1. Load Data (assuming raw CSV without headers for this example)
        df_emg = pd.read_csv(filepath, header=None)
        
        # 2. Standardize Columns (CH_1, CH_2, etc.)
        df_emg.columns = [f'CH_{i+1}' for i in range(df_emg.shape[1])]
        
        # 3. Create Time Vector
        df_emg['Time_s'] = np.arange(len(df_emg)) / fs
        
        # 4. Apply Researcher Pipeline (Example on Channel 1)
        raw_ch1 = df_emg['CH_1']
        
        # Apply Notch
        notch_ch1 = apply_notch_filter(raw_ch1, fs=fs, freq_to_remove=60.0)
        df_emg['CH_1_Notch'] = notch_ch1
        
        # Apply Bandpass (Placeholder)
        bp_ch1 = apply_bandpass_filter(notch_ch1, fs=fs)
        df_emg['CH_1_BP'] = bp_ch1
        
        # Apply Envelope (Placeholder)
        env_ch1 = extract_emg_envelope(bp_ch1, fs=fs)
        df_emg['CH_1_Env'] = env_ch1

        # 5. Comparative Visualization
        plt.figure(figsize=(15, 6))
        plt.plot(df_emg['Time_s'], df_emg['CH_1'], label='Raw Signal (CH_1)', alpha=0.4, color='gray')
        plt.plot(df_emg['Time_s'], df_emg['CH_1_Notch'], label='Filtered (Notch 60Hz)', alpha=0.8, color='blue')
        
        plt.title('EMG Exploration: Automatic Pipeline Application')
        plt.xlabel('Time (seconds)')
        plt.ylabel('Amplitude')
        plt.xlim(0, 5) # Zoom on the first 5 seconds
        plt.legend()
        plt.grid(True)
        plt.show()
        
    except FileNotFoundError:
        print(f"Error: The file {filepath} was not found. Please check your data/raw/ folder.")

# Run the automated pipeline for the first recording found in the DB
if not df_metadata.empty:
    first_record = df_metadata.iloc[0]
    process_and_visualize(first_record)

# inicializar el código inicial necesario

#reconfiguración general de columnas sin título
Una primera intervención con las bases de datos sería agregarles título de columna para facilitar su lectura y para garantizar que no se confundirán con otras bases cuando se avance con la fusión.
Este primer script será para:

_ Configurar los parámetros de la salida si no estuviese hecho ya.

_ Colocar los encabezados de columnas

_ Generar nombres genéricos para las columnas de EMG (ch1, ch2)

_ Crear vectores de tiempo, de ser necesarios

_ Diseño y aplicación de filtro Notch si correspondiese

_ Visualización comparativa de lo realizado.